## Data Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class VICRegTransform:
    def __init__(self):
        # CIFAR-10 images are small, so we use subtle but effective augmentations
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([
                transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

    def __call__(self, x):
        # This returns two different views of the same image 'x'
        x1 = self.transform(x)
        x2 = self.transform(x)
        return x1, x2

# Initialize Dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=VICRegTransform())
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, drop_last=True)

100%|██████████| 170M/170M [00:04<00:00, 34.5MB/s]


## Architecture

In [ ]:
class VICRegModel(nn.Module):
    def __init__(self, embedding_dim=512, projection_dim=2048):
        super().__init__()
        # 1. Encoder (Backbone)
        self.encoder = models.resnet18(weights=None)

        # CIFAR Optimization: replace 7x7 conv with 3x3 and remove maxpool
        self.encoder.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.encoder.maxpool = nn.Identity()
        self.encoder.fc = nn.Identity() # Remove the ImageNet classification head

        # 2. Projector (The Buffer)
        self.projector = nn.Sequential(
            nn.Linear(embedding_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Linear(projection_dim, projection_dim)
        )

    def forward(self, x1, x2):
        # Shared weights: both views pass through the same encoder and projector
        y1 = self.encoder(x1)
        y2 = self.encoder(x2)

        z1 = self.projector(y1)
        z2 = self.projector(y2)
        return z1, z2

## VICReg Loss Function

In [ ]:
def vicreg_loss(z1, z2, sim_coeff=25.0, std_coeff=25.0, cov_coeff=1.0):
    batch_size, num_features = z1.shape

    # 1. Invariance Term: Mean Squared Error
    sim_loss = nn.functional.mse_loss(z1, z2)

    # 2. Variance Term: Hinge loss on standard deviation
    # We subtract the mean of each feature across the batch
    z1 = z1 - z1.mean(dim=0)
    z2 = z2 - z2.mean(dim=0)

    std1 = torch.sqrt(z1.var(dim=0) + 1e-4)
    std2 = torch.sqrt(z2.var(dim=0) + 1e-4)

    std_loss = torch.mean(torch.relu(1 - std1)) + torch.mean(torch.relu(1 - std2))

    # 3. Covariance Term: Decorrelate off-diagonal elements
    def off_diagonal(x):
        n, m = x.shape
        assert n == m
        return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()

    cov1 = (z1.T @ z1) / (batch_size - 1)
    cov2 = (z2.T @ z2) / (batch_size - 1)

    cov_loss = (off_diagonal(cov1).pow(2).sum() / num_features) + \
               (off_diagonal(cov2).pow(2).sum() / num_features)

    return sim_coeff * sim_loss + std_coeff * std_loss + cov_coeff * cov_loss

## Training

In [ ]:
model = VICRegModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-6)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    for batch_idx, ((x1, x2), _) in enumerate(train_loader):
        x1, x2 = x1.to(device), x2.to(device)

        optimizer.zero_grad()
        z1, z2 = model(x1, x2)

        loss = vicreg_loss(z1, z2)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    scheduler.step()
    return total_loss / len(train_loader)

# Run training (e.g., for 100 epochs)
for epoch in range(100):
  train_one_epoch(epoch)

Epoch 0 | Batch 0/195 | Loss: 43.6242
Epoch 0 | Batch 50/195 | Loss: 37.3535
Epoch 0 | Batch 100/195 | Loss: 36.2844
Epoch 0 | Batch 150/195 | Loss: 35.8152
Epoch 1 | Batch 0/195 | Loss: 34.6507
Epoch 1 | Batch 50/195 | Loss: 34.3434
Epoch 1 | Batch 100/195 | Loss: 34.0725
Epoch 1 | Batch 150/195 | Loss: 33.9015
Epoch 2 | Batch 0/195 | Loss: 33.6275
Epoch 2 | Batch 50/195 | Loss: 33.1719
Epoch 2 | Batch 100/195 | Loss: 33.1528
Epoch 2 | Batch 150/195 | Loss: 32.8388
Epoch 3 | Batch 0/195 | Loss: 32.3860
Epoch 3 | Batch 50/195 | Loss: 32.2819
Epoch 3 | Batch 100/195 | Loss: 32.2535
Epoch 3 | Batch 150/195 | Loss: 31.9577
Epoch 4 | Batch 0/195 | Loss: 31.7170
Epoch 4 | Batch 50/195 | Loss: 31.9057
Epoch 4 | Batch 100/195 | Loss: 31.3826
Epoch 4 | Batch 150/195 | Loss: 31.3862
Epoch 5 | Batch 0/195 | Loss: 31.8060
Epoch 5 | Batch 50/195 | Loss: 30.8612
Epoch 5 | Batch 100/195 | Loss: 31.1012
Epoch 5 | Batch 150/195 | Loss: 31.1895
Epoch 6 | Batch 0/195 | Loss: 30.8082
Epoch 6 | Batch 50/1

## Save the model
### because the pre-training was long (4hrs) and I didn't want to redo it

In [ ]:
model_save_path = './vicreg_cifar10_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to ./vicreg_cifar10_model.pth


## Load the model for Linear Probing

In [ ]:
model_save_path = '/content/vicreg_cifar10_model.pth'

# Instantiate a new model with the same architecture
# Make sure to use the same embedding_dim and projection_dim as during training
loaded_model = VICRegModel().to(device)

# Load the state dictionary
loaded_model.load_state_dict(torch.load(model_save_path, map_location=device))

print(f"Model loaded successfully from {model_save_path}")

model = loaded_model

Model loaded successfully from /content/vicreg_cifar10_model.pth


## Attach and Train a simple Linear Classifier and Evaluate

In [ ]:
# 1. Setup Evaluation Data (No augmentations, just normalization)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Original full training dataset
full_train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=eval_transform)

# Calculate 10% of the training data
subset_size = int(0.01 * len(full_train_dataset))
indices = torch.randperm(len(full_train_dataset))[:subset_size]

# Create a subset of the training data
train_subset = torch.utils.data.Subset(full_train_dataset, indices)

train_labeled_loader = DataLoader(
    train_subset, # Use the subset here
    batch_size=256, shuffle=True, num_workers=2
)

test_loader = DataLoader(
    datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform),
    batch_size=256, shuffle=False, num_workers=2
)

def run_linear_probing(vicreg_model, train_loader, test_loader, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # --- Step 1: Freeze Encoder ---
    model.encoder.eval()
    for param in model.encoder.parameters():
        param.requires_grad = False

    # --- Step 2: Initialize Linear Head ---
    # ResNet-18 outputs 512 dimensions
    classifier = nn.Linear(512, 10).to(device)
    optimizer = optim.Adam(classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # --- Step 3: Train Linear Head ---
    print(f"Training Linear Head for {epochs} epochs...")
    for epoch in range(epochs):
        classifier.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.no_grad():
                features = model.encoder(images)

            outputs = classifier(features)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

    # --- Step 4: Final Evaluation ---
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            features = model.encoder(images)
            outputs = classifier(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"\n--- Final Results ---")
    print(f"Linear Probe Accuracy: {accuracy:.2f}%")
    return accuracy

# Execute (Assuming 'model' is your pre-trained model)
accuracy = run_linear_probing(model, train_labeled_loader, test_loader)

Training Linear Head for 50 epochs...
Epoch 5 | Loss: 1.8536
Epoch 10 | Loss: 1.2841
Epoch 15 | Loss: 0.9563
Epoch 20 | Loss: 0.7759
Epoch 25 | Loss: 0.6688
Epoch 30 | Loss: 0.5959
Epoch 35 | Loss: 0.5424
Epoch 40 | Loss: 0.4986
Epoch 45 | Loss: 0.4658
Epoch 50 | Loss: 0.4352

--- Final Results ---
Linear Probe Accuracy: 74.56%


## Simple Linear Classifier Baseline on 1% Labeled Data

To provide a direct comparison, let's train a *truly linear* classifier on the same 1% labeled subset.

In [ ]:
class SimpleLinearClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # CIFAR-10 images are 3x32x32
        input_size = 3 * 32 * 32  # Flattened image size
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.linear(x)
        return x

def train_simple_linear_classifier(train_loader, test_loader, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_linear = SimpleLinearClassifier(num_classes=10).to(device)
    optimizer = optim.Adam(model_linear.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"Training Simple Linear Classifier for {epochs} epochs...")
    for epoch in range(epochs):
        model_linear.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model_linear(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation
    model_linear.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_linear(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"\n--- Simple Linear Classifier Baseline Results ---")
    print(f"Accuracy on Test Set (1% labeled data): {accuracy:.2f}%")
    return accuracy

# Train and evaluate this simple linear baseline
simple_linear_accuracy = train_simple_linear_classifier(train_labeled_loader, test_loader, epochs=50)

Training Simple Linear Classifier for 50 epochs...
Epoch 1 | Loss: 2.3912
Epoch 10 | Loss: 1.0088
Epoch 20 | Loss: 0.6121
Epoch 30 | Loss: 0.4648
Epoch 40 | Loss: 0.4133
Epoch 50 | Loss: 0.4032

--- Simple Linear Classifier Baseline Results ---
Accuracy on Test Set (1% labeled data): 28.92%
